In [5]:
import math
import random
from dataclasses import dataclass
from typing import Dict, List, Tuple

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset


# -----------------------------
# 1) 재현성 + 디바이스
# -----------------------------
def set_seed(seed: int = 42):
    random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

device = "cuda" if torch.cuda.is_available() else "cpu"
set_seed(42)


# -----------------------------
# 2) 합성 이진 분류 데이터 생성
#    (선형 + 비선형 약간 섞어서 2클래스)
# -----------------------------
def make_synthetic_binary(
    n: int = 5000,
    d: int = 20,
    noise: float = 0.3
) -> Tuple[torch.Tensor, torch.Tensor]:
    X = torch.randn(n, d)
    w = torch.randn(d)

    # 약간 비선형 항 추가
    logits = X @ w + 0.5 * torch.sin(X[:, 0]) + 0.25 * (X[:, 1] ** 2)
    logits = logits + noise * torch.randn(n)

    y = (logits > 0).float()  # (n,)
    return X, y

X, y = make_synthetic_binary(n=6000, d=20, noise=0.4)

# train/val split
n_train = 5000
X_train, y_train = X[:n_train], y[:n_train]
X_val, y_val = X[n_train:], y[n_train:]

train_loader = DataLoader(
    TensorDataset(X_train, y_train),
    batch_size=128,
    shuffle=True,
)
val_loader = DataLoader(
    TensorDataset(X_val, y_val),
    batch_size=256,
    shuffle=False,
)


# -----------------------------
# 3) 10개 히든레이어 MLP
#    activation: "relu" 또는 "sigmoid"
# -----------------------------
class DeepMLP(nn.Module):
    def __init__(self, in_dim: int, hidden_dim: int, n_hidden: int, activation: str):
        super().__init__()
        assert n_hidden == 10, "요구사항: 히든 레이어 10개 예시로 고정(원하면 바꿔도 됨)"
        self.activation = activation.lower()

        layers = []
        # 첫 히든
        layers.append(nn.Linear(in_dim, hidden_dim))
        # 나머지 9개 히든
        for _ in range(n_hidden - 1):
            layers.append(nn.Linear(hidden_dim, hidden_dim))
        self.hidden_layers = nn.ModuleList(layers)

        # 출력(로짓 1개)
        self.out = nn.Linear(hidden_dim, 1)

        # (선택) 초기화: 깊은 네트워크라 초기화가 결과에 영향 큼
        self._init_weights()

    def _init_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Linear):
                # ReLU/시그모이드 모두 무난한 Xavier
                nn.init.xavier_uniform_(m.weight)
                nn.init.zeros_(m.bias)

    def act(self, x: torch.Tensor) -> torch.Tensor:
        if self.activation == "relu":
            return F.relu(x)
        elif self.activation == "sigmoid":
            return torch.sigmoid(x)
        else:
            raise ValueError(f"Unknown activation: {self.activation}")

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        for layer in self.hidden_layers:
            x = self.act(layer(x))
        logits = self.out(x).squeeze(-1)  # (batch,)
        return logits


# -----------------------------
# 4) 평가 함수 (loss/acc)
# -----------------------------
@torch.no_grad()
def evaluate(model: nn.Module, loader: DataLoader, criterion: nn.Module) -> Tuple[float, float]:
    model.eval()
    total_loss = 0.0
    total_correct = 0
    total = 0

    for xb, yb in loader:
        xb = xb.to(device)
        yb = yb.to(device)

        logits = model(xb)
        loss = criterion(logits, yb)

        probs = torch.sigmoid(logits)
        preds = (probs >= 0.5).float()

        total_loss += loss.item() * xb.size(0)
        total_correct += (preds == yb).sum().item()
        total += xb.size(0)

    return total_loss / total, total_correct / total


# -----------------------------
# 5) 그래디언트 로깅
#    - 각 레이어 weight grad L2 norm
#    - 전체 grad global norm
# -----------------------------
def grad_stats(model: nn.Module) -> Dict[str, float]:
    stats: Dict[str, float] = {}

    # global grad norm
    sq_sum = 0.0
    for p in model.parameters():
        if p.grad is not None:
            sq_sum += p.grad.detach().pow(2).sum().item()
    stats["grad_global_norm"] = math.sqrt(sq_sum)

    # layer-wise (hidden + out)
    for i, layer in enumerate(model.hidden_layers):
        if layer.weight.grad is not None:
            stats[f"grad_hidden{i+1}_w_norm"] = layer.weight.grad.detach().norm().item()
        if layer.bias.grad is not None:
            stats[f"grad_hidden{i+1}_b_norm"] = layer.bias.grad.detach().norm().item()

    if model.out.weight.grad is not None:
        stats["grad_out_w_norm"] = model.out.weight.grad.detach().norm().item()
    if model.out.bias.grad is not None:
        stats["grad_out_b_norm"] = model.out.bias.grad.detach().norm().item()

    return stats


# -----------------------------
# 6) 학습 루프
# -----------------------------
@dataclass
class TrainConfig:
    in_dim: int = 20
    hidden_dim: int = 64
    n_hidden: int = 10
    lr: float = 1e-3
    epochs: int = 15

def train_one_setting(activation: str, cfg: TrainConfig):
    model = DeepMLP(cfg.in_dim, cfg.hidden_dim, cfg.n_hidden, activation=activation).to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=cfg.lr)
    criterion = nn.BCEWithLogitsLoss()

    history = {
        "train_loss": [],
        "train_acc": [],
        "val_loss": [],
        "val_acc": [],
        "grad_global_norm": [],
        # 필요하면 layer별도 저장 가능
    }

    for epoch in range(1, cfg.epochs + 1):
        model.train()
        running_loss = 0.0
        running_correct = 0
        total = 0

        # (에폭 평균 grad norm 기록용)
        grad_norm_sum = 0.0
        grad_steps = 0

        for xb, yb in train_loader:
            xb = xb.to(device)
            yb = yb.to(device)

            optimizer.zero_grad(set_to_none=True)

            logits = model(xb)
            loss = criterion(logits, yb)
            loss.backward()

            # 그래디언트 통계 (스텝 단위)
            g = grad_stats(model)
            grad_norm_sum += g["grad_global_norm"]
            grad_steps += 1

            optimizer.step()

            # train metrics
            probs = torch.sigmoid(logits.detach())
            preds = (probs >= 0.5).float()
            running_loss += loss.item() * xb.size(0)
            running_correct += (preds == yb).sum().item()
            total += xb.size(0)

        train_loss = running_loss / total
        train_acc = running_correct / total
        val_loss, val_acc = evaluate(model, val_loader, criterion)

        history["train_loss"].append(train_loss)
        history["train_acc"].append(train_acc)
        history["val_loss"].append(val_loss)
        history["val_acc"].append(val_acc)
        history["grad_global_norm"].append(grad_norm_sum / max(1, grad_steps))

        print(
            f"[{activation.upper()}] epoch {epoch:02d} | "
            f"train loss {train_loss:.4f} acc {train_acc:.4f} | "
            f"val loss {val_loss:.4f} acc {val_acc:.4f} | "
            f"avg grad_norm {history['grad_global_norm'][-1]:.4f}"
        )

    return model, history


# -----------------------------
# 7) 실행: ReLU vs Sigmoid 비교
# -----------------------------
cfg = TrainConfig(in_dim=20, hidden_dim=64, n_hidden=10, lr=1e-3, epochs=15)

relu_model, relu_hist = train_one_setting("relu", cfg)
sig_model, sig_hist = train_one_setting("sigmoid", cfg)

print("\n=== Final Comparison (Validation) ===")
print(f"ReLU   val_acc: {relu_hist['val_acc'][-1]:.4f}, val_loss: {relu_hist['val_loss'][-1]:.4f}")
print(f"Sigmoid val_acc: {sig_hist['val_acc'][-1]:.4f}, val_loss: {sig_hist['val_loss'][-1]:.4f}")

print("\nTip) 깊은(10층) + Sigmoid는 gradient vanishing이 더 쉽게 생길 수 있어 "
      "grad_global_norm 추이를 비교해보면 차이가 잘 보일 거야.")

[RELU] epoch 01 | train loss 0.4390 acc 0.8038 | val loss 0.4757 acc 0.8090 | avg grad_norm 1.3667
[RELU] epoch 02 | train loss 0.2038 acc 0.9126 | val loss 0.2025 acc 0.9160 | avg grad_norm 2.2580
[RELU] epoch 03 | train loss 0.1485 acc 0.9402 | val loss 0.1776 acc 0.9320 | avg grad_norm 1.3883
[RELU] epoch 04 | train loss 0.1101 acc 0.9518 | val loss 0.1563 acc 0.9370 | avg grad_norm 1.1826
[RELU] epoch 05 | train loss 0.0939 acc 0.9598 | val loss 0.1518 acc 0.9370 | avg grad_norm 1.2499
[RELU] epoch 06 | train loss 0.0784 acc 0.9690 | val loss 0.1669 acc 0.9300 | avg grad_norm 1.4045
[RELU] epoch 07 | train loss 0.0688 acc 0.9722 | val loss 0.1617 acc 0.9450 | avg grad_norm 1.3154
[RELU] epoch 08 | train loss 0.0557 acc 0.9790 | val loss 0.1678 acc 0.9440 | avg grad_norm 1.4417
[RELU] epoch 09 | train loss 0.0445 acc 0.9802 | val loss 0.2227 acc 0.9430 | avg grad_norm 1.4841
[RELU] epoch 10 | train loss 0.0307 acc 0.9884 | val loss 0.1971 acc 0.9500 | avg grad_norm 1.2365
[RELU] epo